In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Ejercicio 1

**Modelo de mezcla de gaussianas.** Cada observación tiene una etiqueta latente $z_i\in\{1,\dots,K\}$:

$$
z_i\sim\text{Multinomial}(1,\boldsymbol\phi),\qquad \mathbf x_i\mid z_i=k\ \sim\ \mathcal N(\boldsymbol\mu_k,\Sigma_k),
$$

con $\boldsymbol\theta=(\boldsymbol\phi,\boldsymbol\mu_1,\dots,\boldsymbol\mu_K,\Sigma_1,\dots,\Sigma_K)$. La log-verosimilitud de los datos observados es

$$
\ell(\boldsymbol\theta)=\sum_{i=1}^n\log\sum_{k=1}^K\phi_k\,\mathcal N(\mathbf x_i;\boldsymbol\mu_k,\Sigma_k).
$$

**Algoritmo EM.** A partir de $\boldsymbol\theta^{(0)}$, repetir:

1. **Paso E** (responsabilidades):
$$
w_{ik}^{(t)}=p(z_i=k\mid\mathbf x_i;\boldsymbol\theta^{(t)})=\frac{\phi_k\,\mathcal N(\mathbf x_i;\boldsymbol\mu_k,\Sigma_k)}{\sum_{l=1}^K\phi_l\,\mathcal N(\mathbf x_i;\boldsymbol\mu_l,\Sigma_l)} .
$$
2. **Paso M** (promedios ponderados por las responsabilidades), con $N_k^{(t)}=\sum_i w_{ik}^{(t)}$:
$$
\phi_k^{(t+1)}=\frac{N_k^{(t)}}{n},\qquad
\boldsymbol\mu_k^{(t+1)}=\frac{1}{N_k^{(t)}}\sum_{i=1}^n w_{ik}^{(t)}\,\mathbf x_i,\qquad
\Sigma_k^{(t+1)}=\frac{1}{N_k^{(t)}}\sum_{i=1}^n w_{ik}^{(t)}\,(\mathbf x_i-\boldsymbol\mu_k^{(t+1)})(\mathbf x_i-\boldsymbol\mu_k^{(t+1)})^\top .
$$

**Criterio de parada:** el incremento $\ell(\boldsymbol\theta^{(t+1)})-\ell(\boldsymbol\theta^{(t)})$ es menor que una tolerancia (o se alcanza `max_iter`, y en ese caso se avisa con `warnings.warn`).

**Inicialización:** medias en $K$ observaciones elegidas al azar, $\Sigma_k=S$ (covarianza muestral de todo el conjunto) y $\phi_k=1/K$.

**Cálculo estable.** Trabajamos con logaritmos:

$$
\log\mathcal N(\mathbf x;\boldsymbol\mu,\Sigma)=-\tfrac12\left[p\log(2\pi)+\log\det\Sigma+(\mathbf x-\boldsymbol\mu)^\top\Sigma^{-1}(\mathbf x-\boldsymbol\mu)\right].
$$

Con $a_{ik}=\log\phi_k+\log\mathcal N(\mathbf x_i;\boldsymbol\mu_k,\Sigma_k)$ y $m_i=\max_l a_{il}$:

$$
\log\sum_{l}e^{a_{il}}=m_i+\log\sum_l e^{a_{il}-m_i},\qquad \log w_{ik}=a_{ik}-\log\sum_l e^{a_{il}} .
$$

Después de restar $m_i$, el mayor exponente es $0$: la suma vale al menos $1$ y no puede dar $0$. Sumando $\log\sum_l e^{a_{il}}$ sobre $i$ obtenemos $\ell(\boldsymbol\theta)$ sin cálculo adicional.

**Regularización.** Si una componente se "cierra" sobre muy pocos puntos, su $\Sigma_k$ se vuelve casi singular y $\ell\to\infty$. Como hace scikit-learn (`reg_covar`), sumamos $10^{-6}I$ a cada $\Sigma_k$ en el paso M.

Como EM solo garantiza un máximo local (ítem b), el parámetro `n_init` repite el ajuste con `n_init` inicializaciones distintas y conserva la de mayor $\ell$, igual que en `myKMeans` (guía 3.2) y en `GaussianMixture`.

¿Por qué hace falta? Un ejemplo con números del orden de los que aparecen en $\mathbb R^{p}$ con $p$ moderado:

In [ ]:
a = np.array([-800.0, -805.0])  # log(φ_k N(x; μ_k, Σ_k)) para dos componentes

# Directo: las densidades dan 0 (underflow) y el cociente es 0/0
dens = np.exp(a)
print("densidades:", dens)
# print("w directo:", dens / dens.sum())

# Con log-sum-exp
m = np.max(a)
log_suma = m + np.log(np.sum(np.exp(a - m)))
print("w log-sum-exp:", np.exp(a - log_suma))

In [ ]:
class myGMM:
    """
    Modelo de mezcla de gaussianas ajustado con el algoritmo EM.
    """
    def __init__(self, K, max_iter=500, tol=1e-6, reg_covar=1e-6, n_init=1, random_state=None):
        """
        Inicializa el modelo.

        Parámetros:
        - K : int
            Número de componentes
        - max_iter : int
            Número máximo de iteraciones
        - tol : float
            Tolerancia sobre el incremento de la log-verosimilitud
        - reg_covar : float
            Constante que se suma a la diagonal de cada Σ_k
        - n_init : int
            Cantidad de inicializaciones (se conserva la de mayor log-verosimilitud)
        - random_state : int o None
            Semilla para la inicialización
        """

        self.K = K
        self.max_iter = max_iter
        self.tol = tol
        self.reg_covar = reg_covar
        self.n_init = n_init
        self.random_state = random_state

        # Resultados del ajuste
        self.phi = None  # pesos (K,)
        self.mu = None  # medias (K, p)
        self.Sigma = None  # covarianzas (K, p, p)
        self.W = None  # responsabilidades (n, K)
        self.etiquetas = None  # componente de máxima responsabilidad (n,)
        self.ell = None  # log-verosimilitud final
        self.ell_hist = None  # log-verosimilitud en cada iteración
        self.n_iter_ = None  # iteraciones de la mejor corrida

    def log_gaussiana(self, X, mu, Sigma):
        """
        log N(x_i; μ, Σ) para cada observación.

        Parámetros:
        - X: datos (n, p)
        - mu: media (p,)
        - Sigma: covarianza (p, p)

        Retorna:
        - log_dens: (n,)
        """

        # TODO:
        # - Calcular Σ^{-1} y log det Σ (np.linalg.slogdet)
        # - Calcular la distancia de Mahalanobis de cada x_i a μ
        # - Devolver -1/2 [p log(2π) + log det Σ + Mahalanobis]

        n, p = X.shape
        Sigma_inv = ...
        _, logdet = ...
        D = ...  # (n, p)

        # Opción 1: muestra a muestra
        maha = np.zeros(n)
        for i in range(n):
            maha[i] = ...

        # Opción 2: vectorizada
        maha = ...  # (n,)

        return ...

    def paso_E(self, X, phi, mu, Sigma):
        """
        Paso E: responsabilidades w_ik y log-verosimilitud ℓ(θ).

        Parámetros:
        - X: datos (n, p)
        - phi: (K,), mu: (K, p), Sigma: (K, p, p)

        Retorna:
        - W: responsabilidades (n, K)
        - ell: log-verosimilitud
        """

        # TODO:
        # - Armar la matriz A con a_ik = log φ_k + log N(x_i; μ_k, Σ_k)
        # - Calcular log Σ_l exp(a_il) restando el máximo de cada fila
        # - W = exp(A - log Σ_l exp(a_il)),  ell = suma de log Σ_l exp(a_il)

        n = X.shape[0]
        A = np.zeros((n, self.K))  # (n, K)
        for k in range(self.K):
            # a_ik = log(φ_k N(x_i; μ_k, Σ_k)) = log p(x_i, z_i = k): log de la conjunta
            A[:, k] = ...

        # log-sum-exp: log Σ_l exp(a_il) = m_i + log Σ_l exp(a_il - m_i),  con m_i = max_l a_il
        # Restar m_i no cambia el resultado (es una identidad exacta), pero evita el underflow:
        # el mayor exponente pasa a ser 0, la suma queda >= 1 y su log está bien definido
        # log_suma_i = log p(x_i; θ): log de la marginal
        m = ...  # (n, 1): máximo de cada fila
        log_suma = ...  # (n, 1)

        # Bayes: w_ik = conjunta / marginal = exp(a_ik - log p(x_i; θ))
        W = ...  # (n, K)
        ell = ...  # ℓ(θ) = Σ_i log p(x_i; θ)

        return W, ell

    def paso_M(self, X, W):
        """
        Paso M: actualización de φ, μ_k y Σ_k.

        Parámetros:
        - X: datos (n, p)
        - W: responsabilidades (n, K)

        Retorna:
        - phi: (K,), mu: (K, p), Sigma: (K, p, p)
        """

        # TODO:
        # - N_k = Σ_i w_ik
        # - φ_k = N_k / n,  μ_k = Σ_i w_ik x_i / N_k
        # - Σ_k = Σ_i w_ik (x_i - μ_k)(x_i - μ_k)^T / N_k  (+ reg_covar I)

        n, p = X.shape
        Nk = ...  # (K,)
        phi = ...  # (K,)

        # Opción 1: componente a componente
        mu = np.zeros((self.K, p))
        for k in range(self.K):
            # (n, 1) * (n, p) -> (n, p) -> suma_n -> (p,)
            mu[k] = ...

        # Opción 2: vectorizada
        # (K, n) @ (n, p) / (K, 1) -> (K, p)
        mu = ...  # (K, p)

        Sigma = np.zeros((self.K, p, p))
        for k in range(self.K):
            D = ...  # (n, p)

            # Opción 1: muestra a muestra
            for i in range(n):
                Sigma[k] += ...  # (p, p)
            Sigma[k] /= ...

            # Opción 2: vectorizada
            # (p, n) @ (n, p) / N_k -> (p, p)
            Sigma[k] = ...

            Sigma[k] += self.reg_covar * np.eye(p)

        return phi, mu, Sigma

    def fit(self, X):
        """
        Ajusta el modelo n_init veces, alternando los pasos E y M, y conserva la corrida de mayor ℓ.

        Parámetros:
        - X: datos (n, p)

        Retorna:
        - self
        """

        # TODO:
        # - Repetir n_init veces (con el mismo generador rng):
        #   - inicializar: φ_k = 1/K, μ_k = K observaciones al azar, Σ_k = covarianza muestral de X
        #   - paso E inicial (W y ℓ(θ^(0)))
        #   - alternar paso M y paso E, guardando ℓ en cada iteración
        #   - detenerse si el incremento de ℓ es menor que tol; si se llega a max_iter, avisar
        # - Conservar la corrida de mayor ℓ final

        rng = np.random.default_rng(self.random_state)
        X = np.asarray(X, dtype=float)
        n, p = X.shape

        # Σ_k inicial: covarianza muestral de X + reg_covar I (la misma en todas las corridas)
        Xc = X - X.mean(axis=0)  # (n, p)
        # (p, n) @ (n, p) / n -> (p, p)
        S = (... @ ...) / ... + self.reg_covar * ...  # (p, p)

        # Opcional con np.cov (rowvar=False para que las columnas sean variables, bias=True para dividir por n)
        # S = np.cov(X, rowvar=False, bias=True) + self.reg_covar * np.eye(p)  # (p, p)

        self.ell = -np.inf
        for _ in range(self.n_init):
            phi = ...  # (K,)
            indices = ...
            mu = ...  # (K, p)
            Sigma = np.array([S.copy() for _ in range(self.K)])  # (K, p, p)

            # Paso E inicial
            W, ell = ...
            ell_hist = [ell]

            for t in range(self.max_iter):
                # Paso M
                phi, mu, Sigma = ...

                # Paso E (responsabilidades y ℓ con los parámetros nuevos)
                W, ell_nuevo = ...
                ell_hist.append(ell_nuevo)

                # Comprobación de convergencia
                incremento = ...
                ell = ell_nuevo
                if ...:
                    break
            else:
                warnings.warn(f"myGMM no convergió en {self.max_iter} iteraciones")

            if ...:  # la mejor corrida hasta ahora
                self.phi, self.mu, self.Sigma = phi, mu, Sigma
                self.W = W
                self.etiquetas = ...
                self.ell = ell
                self.ell_hist = ell_hist
                self.n_iter_ = t + 1

        return self

    def predict_proba(self, X):
        """
        Responsabilidades de cada observación de X (n, K).
        """

        W, _ = self.paso_E(np.asarray(X, dtype=float), self.phi, self.mu, self.Sigma)

        return W

    def predict(self, X):
        """
        Componente de máxima responsabilidad de cada observación de X.
        """

        return np.argmax(self.predict_proba(X), axis=1)

##### Solución

In [ ]:
class myGMM:
    """
    Modelo de mezcla de gaussianas ajustado con el algoritmo EM.
    """
    def __init__(self, K, max_iter=500, tol=1e-6, reg_covar=1e-6, n_init=1, random_state=None):
        """
        Inicializa el modelo.

        Parámetros:
        - K : int
            Número de componentes
        - max_iter : int
            Número máximo de iteraciones
        - tol : float
            Tolerancia sobre el incremento de la log-verosimilitud
        - reg_covar : float
            Constante que se suma a la diagonal de cada Σ_k
        - n_init : int
            Cantidad de inicializaciones (se conserva la de mayor log-verosimilitud)
        - random_state : int o None
            Semilla para la inicialización
        """

        self.K = K
        self.max_iter = max_iter
        self.tol = tol
        self.reg_covar = reg_covar
        self.n_init = n_init
        self.random_state = random_state

        # Resultados del ajuste
        self.phi = None  # pesos (K,)
        self.mu = None  # medias (K, p)
        self.Sigma = None  # covarianzas (K, p, p)
        self.W = None  # responsabilidades (n, K)
        self.etiquetas = None  # componente de máxima responsabilidad (n,)
        self.ell = None  # log-verosimilitud final
        self.ell_hist = None  # log-verosimilitud en cada iteración
        self.n_iter_ = None  # iteraciones de la mejor corrida

    def log_gaussiana(self, X, mu, Sigma):
        """
        log N(x_i; μ, Σ) para cada observación.

        Parámetros:
        - X: datos (n, p)
        - mu: media (p,)
        - Sigma: covarianza (p, p)

        Retorna:
        - log_dens: (n,)
        """

        n, p = X.shape
        Sigma_inv = np.linalg.inv(Sigma)
        _, logdet = np.linalg.slogdet(Sigma)
        D = X - mu  # (n, p)

        # Opción 1: muestra a muestra
        maha = np.zeros(n)
        for i in range(n):
            maha[i] = D[i] @ Sigma_inv @ D[i]

        # Opción 2: vectorizada
        maha = np.sum((D @ Sigma_inv) * D, axis=1)  # (n,)

        return -0.5 * (p * np.log(2 * np.pi) + logdet + maha)

    def paso_E(self, X, phi, mu, Sigma):
        """
        Paso E: responsabilidades w_ik y log-verosimilitud ℓ(θ).

        Parámetros:
        - X: datos (n, p)
        - phi: (K,), mu: (K, p), Sigma: (K, p, p)

        Retorna:
        - W: responsabilidades (n, K)
        - ell: log-verosimilitud
        """

        n = X.shape[0]
        A = np.zeros((n, self.K))  # (n, K)
        for k in range(self.K):
            # a_ik = log(φ_k N(x_i; μ_k, Σ_k)) = log p(x_i, z_i = k): log de la conjunta
            A[:, k] = np.log(phi[k]) + self.log_gaussiana(X, mu[k], Sigma[k])

        # log-sum-exp: log Σ_l exp(a_il) = m_i + log Σ_l exp(a_il - m_i),  con m_i = max_l a_il
        # Restar m_i no cambia el resultado (es una identidad exacta), pero evita el underflow:
        # el mayor exponente pasa a ser 0, la suma queda >= 1 y su log está bien definido
        # log_suma_i = log p(x_i; θ): log de la marginal
        m = np.max(A, axis=1, keepdims=True)  # (n, 1): máximo de cada fila
        log_suma = m + np.log(np.sum(np.exp(A - m), axis=1, keepdims=True))  # (n, 1)

        # Bayes: w_ik = conjunta / marginal = exp(a_ik - log p(x_i; θ))
        W = np.exp(A - log_suma)  # (n, K)
        ell = np.sum(log_suma)  # ℓ(θ) = Σ_i log p(x_i; θ)

        return W, ell

    def paso_M(self, X, W):
        """
        Paso M: actualización de φ, μ_k y Σ_k.

        Parámetros:
        - X: datos (n, p)
        - W: responsabilidades (n, K)

        Retorna:
        - phi: (K,), mu: (K, p), Sigma: (K, p, p)
        """

        n, p = X.shape
        Nk = np.sum(W, axis=0)  # (K,)
        phi = Nk / n  # (K,)

        # Opción 1: componente a componente
        mu = np.zeros((self.K, p))
        for k in range(self.K):
            # (n, 1) * (n, p) -> (n, p) -> suma_n -> (p,)
            mu[k] = np.sum(W[:, k, None] * X, axis=0) / Nk[k]

        # Opción 2: vectorizada
        # (K, n) @ (n, p) / (K, 1) -> (K, p)
        mu = (W.T @ X) / Nk[:, None]  # (K, p)

        Sigma = np.zeros((self.K, p, p))
        for k in range(self.K):
            D = X - mu[k]  # (n, p)

            # Opción 1: muestra a muestra
            for i in range(n):
                Sigma[k] += W[i, k] * np.outer(D[i], D[i])  # (p, p)
            Sigma[k] /= Nk[k]

            # Opción 2: vectorizada
            # (p, n) @ (n, p) / N_k -> (p, p)
            Sigma[k] = (D.T @ (W[:, k, None] * D)) / Nk[k]

            Sigma[k] += self.reg_covar * np.eye(p)

        return phi, mu, Sigma

    def fit(self, X):
        """
        Ajusta el modelo n_init veces, alternando los pasos E y M, y conserva la corrida de mayor ℓ.

        Parámetros:
        - X: datos (n, p)

        Retorna:
        - self
        """

        rng = np.random.default_rng(self.random_state)
        X = np.asarray(X, dtype=float)
        n, p = X.shape

        # Σ_k inicial: covarianza muestral de X + reg_covar I (la misma en todas las corridas)
        Xc = X - X.mean(axis=0)  # (n, p)
        # (p, n) @ (n, p) / n -> (p, p)
        S = (Xc.T @ Xc) / n + self.reg_covar * np.eye(p)  # (p, p)

        # Opcional con np.cov (rowvar=False para que las columnas sean variables, bias=True para dividir por n)
        # S = np.cov(X, rowvar=False, bias=True) + self.reg_covar * np.eye(p)  # (p, p)

        self.ell = -np.inf
        for _ in range(self.n_init):
            phi = np.ones(self.K) / self.K  # (K,)
            indices = rng.choice(n, self.K, replace=False)
            mu = X[indices]  # (K, p)
            Sigma = np.array([S.copy() for _ in range(self.K)])  # (K, p, p)

            # Paso E inicial
            W, ell = self.paso_E(X, phi, mu, Sigma)
            ell_hist = [ell]

            for t in range(self.max_iter):
                # Paso M
                phi, mu, Sigma = self.paso_M(X, W)

                # Paso E (responsabilidades y ℓ con los parámetros nuevos)
                W, ell_nuevo = self.paso_E(X, phi, mu, Sigma)
                ell_hist.append(ell_nuevo)

                # Comprobación de convergencia
                incremento = ell_nuevo - ell
                ell = ell_nuevo
                if incremento < self.tol:
                    break
            else:
                warnings.warn(f"myGMM no convergió en {self.max_iter} iteraciones")

            if ell > self.ell:  # la mejor corrida hasta ahora
                self.phi, self.mu, self.Sigma = phi, mu, Sigma
                self.W = W
                self.etiquetas = np.argmax(W, axis=1)
                self.ell = ell
                self.ell_hist = ell_hist
                self.n_iter_ = t + 1

        return self

    def predict_proba(self, X):
        """
        Responsabilidades de cada observación de X (n, K).
        """

        W, _ = self.paso_E(np.asarray(X, dtype=float), self.phi, self.mu, self.Sigma)

        return W

    def predict(self, X):
        """
        Componente de máxima responsabilidad de cada observación de X.
        """

        return np.argmax(self.predict_proba(X), axis=1)

##### Funciones auxiliares

Funciones auxiliares para graficar. `graficar_clusters` es la de la guía anterior. `graficar_elipses` dibuja, para cada componente, las curvas de nivel $(\mathbf x-\boldsymbol\mu_k)^\top\Sigma_k^{-1}(\mathbf x-\boldsymbol\mu_k)=c^2$ con $c=1,2$. Si $\Sigma_k=V\Lambda V^\top$ (autovalores y autovectores), es una elipse centrada en $\boldsymbol\mu_k$, con ejes en las direcciones de los autovectores y semiejes $c\sqrt{\lambda}$.

In [ ]:
from matplotlib.patches import Ellipse

COLORES = plt.cm.tab10.colors

def graficar_clusters(X, etiquetas, ax, centroides=None, titulo=""):
    """
    Grafica un agrupamiento en R^2.

    Parámetros:
    - X: datos (n, 2)
    - etiquetas: (n,), con -1 para el ruido
    - ax: eje de matplotlib
    - centroides: (K, 2) o None
    """
    etiquetas = np.asarray(etiquetas)
    ruido = etiquetas == -1

    colores = [COLORES[e % len(COLORES)] for e in etiquetas[~ruido]]
    ax.scatter(X[~ruido, 0], X[~ruido, 1], c=colores, s=10)

    if ruido.any():
        ax.scatter(X[ruido, 0], X[ruido, 1], c="k", marker="x", s=15, linewidths=0.8, label="ruido")
        ax.legend(loc="best", fontsize=8)

    if centroides is not None:
        ax.scatter(centroides[:, 0], centroides[:, 1], c="k", marker="X", s=150, edgecolors="w", linewidths=1.5)

    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")


def graficar_elipses(ax, mu, Sigma, n_std=(1, 2)):
    """
    Elipses de covarianza de cada componente en R^2.

    Parámetros:
    - ax: eje de matplotlib
    - mu: medias (K, 2)
    - Sigma: covarianzas (K, 2, 2)
    - n_std: valores de c (distancia de Mahalanobis) a graficar
    """
    for k in range(len(mu)):
        valores, vectores = np.linalg.eigh(Sigma[k])  # autovalores en orden creciente
        angulo = np.degrees(np.arctan2(vectores[1, 1], vectores[0, 1]))  # dirección del eje mayor
        for c in n_std:
            ancho, alto = 2 * c * np.sqrt(valores[::-1])
            ax.add_patch(Ellipse(mu[k], ancho, alto, angle=angulo, fill=False,
                                 color=COLORES[k % len(COLORES)], lw=1.5, alpha=0.9))

#### a) Verificación con `make_blobs`

Generamos tres grupos con dispersiones distintas para que las covarianzas estimadas no sean todas iguales.

In [ ]:
from sklearn.datasets import make_blobs

X_b, y_b = make_blobs(n_samples=600, centers=3, cluster_std=[1.0, 2.0, 0.5], random_state=4)

gmm = myGMM(K=3, random_state=0).fit(X_b)

print(f"Iteraciones: {gmm.n_iter_}, ℓ final = {gmm.ell:.2f}\n")
for k in range(gmm.K):
    print(f"Componente {k}: φ = {gmm.phi[k]:.3f}, μ = {np.round(gmm.mu[k], 2)}, "
          f"Σ = {np.round(gmm.Sigma[k], 2).tolist()}")

Miramos las responsabilidades de algunas observaciones:

- una **entre dos grupos**: la de menor responsabilidad máxima $\max_k w_{ik}$, es decir, la más "dudosa";
- una **central** de cada componente: la más cercana (en distancia de Mahalanobis) a $\boldsymbol\mu_k$.

In [ ]:
W = gmm.W

# Observación entre grupos: la de menor responsabilidad máxima
i_frontera = np.argmin(np.max(W, axis=1))

# Observación central de cada componente: la de menor distancia de Mahalanobis a μ_k
i_centrales = []
for k in range(gmm.K):
    D = X_b - gmm.mu[k]
    maha = np.sum((D @ np.linalg.inv(gmm.Sigma[k])) * D, axis=1)
    i_centrales.append(np.argmin(maha))

np.set_printoptions(precision=4, suppress=True)
print(f"Entre grupos: x = {X_b[i_frontera].round(2)}, w = {W[i_frontera]}")
for k, i in enumerate(i_centrales):
    print(f"Central de la componente {k}: x = {X_b[i].round(2)}, w = {W[i]}")

fig, ax = plt.subplots(figsize=(7, 7))
graficar_clusters(X_b, gmm.etiquetas, ax, gmm.mu, titulo=f"myGMM, K=3 ({gmm.n_iter_} iter.)")
graficar_elipses(ax, gmm.mu, gmm.Sigma)
for i, nombre in [(i_frontera, "entre grupos")] + [(i, f"central {k}") for k, i in enumerate(i_centrales)]:
    ax.scatter(*X_b[i], s=200, facecolors="none", edgecolors="k", linewidths=2)
    ax.annotate(nombre, X_b[i], xytext=(12, -12), textcoords="offset points", fontsize=10)
ax.set_aspect("equal")
plt.show()

#### b) Verificación de la monotonía

Usamos un conjunto con 5 grupos (el de la guía anterior, en el que K-medias quedaba atrapado en malos mínimos locales) y ajustamos con 10 inicializaciones distintas.

In [ ]:
X_5, y_5 = make_blobs(n_samples=500, centers=5, cluster_std=1.0, random_state=7)

modelos_b = [myGMM(K=5, random_state=s).fit(X_5) for s in range(10)]

fig, ax = plt.subplots(figsize=(8, 4.5))
for s, m in enumerate(modelos_b):
    ax.plot(m.ell_hist, label=f"semilla {s}")
ax.set_xlabel("iteración t")
ax.set_ylabel(r"$\ell(\theta^{(t)})$")
ax.set_title("Log-verosimilitud a lo largo de las iteraciones")
ax.set_ylim(np.min([m.ell_hist[1] for m in modelos_b]) - 50, None)
ax.legend(fontsize=8, ncol=2)
plt.show()

filas = []
for s, m in enumerate(modelos_b):
    dif = np.diff(m.ell_hist)
    filas.append({"semilla": s, "iteraciones": m.n_iter_, "ℓ final": round(m.ell, 2), "¿no decreciente?": np.all(dif >= -1e-8), "menor incremento": f"{dif.min():.2e}"})
pd.DataFrame(filas).set_index("semilla")

In [ ]:
ell_finales = np.array([m.ell for m in modelos_b])
mejor = modelos_b[np.argmax(ell_finales)]
peor = modelos_b[np.argmin(ell_finales)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, m, txt in [(axes[0], mejor, "Mayor"), (axes[1], peor, "Menor")]:
    graficar_clusters(X_5, m.etiquetas, ax, m.mu, titulo=f"{txt} ℓ = {m.ell:.1f}")
    graficar_elipses(ax, m.mu, m.Sigma)
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

**Para discutir:** ¿Son no decrecientes todas las curvas? ¿Llegan todas al mismo valor de $\ell$?

Por eso `myGMM` tiene el parámetro `n_init`: con `n_init=10`, cada ajuste prueba 10 inicializaciones y conserva la de mayor $\ell$. Lo usamos en los ítems siguientes.

In [ ]:
gmm_n = myGMM(K=5, n_init=10, random_state=0).fit(X_5)
print(f"Mayor ℓ entre las 10 semillas de arriba: {ell_finales.max():.2f}")
print(f"myGMM con n_init=10: ℓ = {gmm_n.ell:.2f}")

#### c) Clusters no esféricos

Generamos `X_elip` con la misma función de la guía anterior (`generar_X_elip`, con semillas fijas) y comparamos con K-medias. Para no repetir la clase `myKMeans`, usamos `KMeans` de scikit-learn con `n_init=10`, que llega a la misma solución ($J$ igual al obtenido con `myKMeans`). Para cuantificar el acuerdo con los grupos verdaderos usamos el índice de Rand ajustado (ARI): vale $1$ si las particiones coinciden (salvo renumeración de los clusters) y $\approx0$ si el acuerdo es el que se esperaría por azar.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

def generar_X_elip():
    """
    Cuatro grupos gaussianos deformados por una transformación lineal (los datos de la guía 3.2).

    Retorna:
    - X_elip: (400, 2)
    - y_true: grupo de cada observación (400,)
    """
    X_raw, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.60, random_state=0)
    rng = np.random.RandomState(13)
    return X_raw @ rng.normal(size=(2, 2)), y_true


X_elip, y_true = generar_X_elip()

km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X_elip)
gmm_elip = myGMM(K=4, n_init=10, random_state=0).fit(X_elip)

mu_true = np.array([X_elip[y_true == k].mean(axis=0) for k in range(4)])
Sigma_true = np.array([np.cov(X_elip[y_true == k], rowvar=False) for k in range(4)])

print(f"K-medias: J = {km.inertia_:.2f}, ARI = {adjusted_rand_score(y_true, km.labels_):.3f}")
print(f"GMM: ℓ = {gmm_elip.ell:.2f}, ARI = {adjusted_rand_score(y_true, gmm_elip.etiquetas):.3f}")

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
graficar_clusters(X_elip, y_true, axes[0], mu_true, titulo="Grupos verdaderos")
graficar_elipses(axes[0], mu_true, Sigma_true)
graficar_clusters(X_elip, km.labels_, axes[1], km.cluster_centers_, titulo="K-medias, K=4")
graficar_clusters(X_elip, gmm_elip.etiquetas, axes[2], gmm_elip.mu, titulo="myGMM, K=4")
graficar_elipses(axes[2], gmm_elip.mu, gmm_elip.Sigma)
for ax in axes:
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

**Para discutir:** ¿Qué estima GMM que K-medias no, y por qué eso le permite recuperar los grupos elongados?

#### d) Comparación con `GaussianMixture` de scikit-learn

Para que la comparación sea justa, le pasamos a `GaussianMixture` la **misma inicialización** que usa `myGMM` (mismo generador y misma semilla):

- `weights_init`: $\phi_k=1/K$;
- `means_init`: las mismas $K$ observaciones;
- `precisions_init`: las inversas de las covarianzas iniciales ($S^{-1}$).

Por defecto, scikit-learn inicializa las responsabilidades con K-medias. Además:

- su criterio de parada usa el incremento de $\ell/n$ (log-verosimilitud **promedio**), no de $\ell$;
- `reg_covar=1e-6` coincide con el nuestro;
- `score(X)` devuelve $\ell/n$.

Usamos tolerancias chicas en ambos para comparar los puntos fijos.

In [ ]:
from sklearn.mixture import GaussianMixture

K, semilla = 3, 0
gmm = myGMM(K=K, tol=1e-10, random_state=semilla).fit(X_b)

# Misma inicialización que myGMM
n, p = X_b.shape
idx0 = np.random.default_rng(semilla).choice(n, size=K, replace=False)
S0 = np.cov(X_b, rowvar=False, bias=True) + 1e-6 * np.eye(p)

gm_sk = GaussianMixture(
    n_components=K,
    covariance_type="full",
    tol=1e-12,
    max_iter=1000,
    weights_init=np.full(K, 1 / K),
    means_init=X_b[idx0],
    precisions_init=np.array([np.linalg.inv(S0)] * K),
).fit(X_b)

print(f"Iteraciones: myGMM {gmm.n_iter_}, sklearn {gm_sk.n_iter_}")
print(f"ℓ: myGMM {gmm.ell:.6f}, sklearn {gm_sk.score(X_b) * n:.6f}")
print(f"máx. |Δφ| = {np.max(np.abs(gmm.phi - gm_sk.weights_)):.2e}")
print(f"máx. |Δμ| = {np.max(np.abs(gmm.mu - gm_sk.means_)):.2e}")
print(f"máx. |ΔΣ| = {np.max(np.abs(gmm.Sigma - gm_sk.covariances_)):.2e}")
print(f"máx. |ΔW| = {np.max(np.abs(gmm.W - gm_sk.predict_proba(X_b))):.2e}")
print(f"Etiquetas iguales: {np.array_equal(gmm.etiquetas, gm_sk.predict(X_b))}")

Con la configuración por defecto (inicialización con K-medias), scikit-learn puede numerar las componentes de otra forma. Las particiones se comparan entonces con el ARI, que no depende de la numeración:

In [ ]:
gm_def = GaussianMixture(n_components=3, random_state=0).fit(X_b)

print(f"ℓ (sklearn por defecto) = {gm_def.score(X_b) * n:.4f}")
print(f"ARI entre myGMM y sklearn por defecto: {adjusted_rand_score(gmm.etiquetas, gm_def.predict(X_b)):.4f}")
print("pesos sklearn:", np.sort(gm_def.weights_), "| pesos myGMM:", np.sort(gmm.phi))

## Ejercicio 2 *(Adicional)*

Rearmamos la grilla de la guía anterior: los mismos cinco conjuntos estandarizados y los mismos hiperparámetros. Para no repetir `myKMeans` y `myDBSCAN`, usamos `KMeans` y `DBSCAN` de scikit-learn (en la guía 3.2 verificamos que dan los mismos resultados). Esta vez guardamos también las etiquetas verdaderas, para calcular el ARI de cada método.

Para GMM usamos `myGMM` con `n_init=10` con el mismo $K$ que K-medias, y graficamos también las elipses. Para eso, la función de cada método puede devolver un cuarto elemento opcional con las covarianzas.

In [ ]:
from sklearn.datasets import make_circles, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

n = 500
rng = np.random.default_rng(0)

datos_grilla = {
    "blobs (2 centros)": make_blobs(n_samples=n, centers=2, random_state=8),
    "blobs (5 centros)": make_blobs(n_samples=n, centers=5, cluster_std=0.8, random_state=3),
    "circles": make_circles(n_samples=n, factor=0.5, noise=0.05, random_state=0),
    "moons": make_moons(n_samples=n, noise=0.05, random_state=0),
    "uniforme": (rng.uniform(-1, 1, size=(n, 2)), None),
}
datasets = {nombre: StandardScaler().fit_transform(X) for nombre, (X, _) in datos_grilla.items()}
y_grilla = {nombre: y for nombre, (_, y) in datos_grilla.items()}

nu = 5
hiper = {
    "blobs (2 centros)": {"K": 2, "eps": 0.40, "nu": nu},
    "blobs (5 centros)": {"K": 5, "eps": 0.15, "nu": nu},
    "circles":           {"K": 2, "eps": 0.25, "nu": nu},
    "moons":             {"K": 2, "eps": 0.25, "nu": nu},
    "uniforme":          {"K": 3, "eps": 0.30, "nu": nu},
}

In [ ]:
def ajustar_kmeans(nombre, X):
    km = KMeans(n_clusters=hiper[nombre]["K"], n_init=10, random_state=0).fit(X)
    return km.labels_, km.cluster_centers_, f"K = {km.n_clusters}"

def ajustar_dbscan(nombre, X):
    db = DBSCAN(eps=hiper[nombre]["eps"], min_samples=hiper[nombre]["nu"]).fit(X)
    et = db.labels_
    return et, None, f"ε = {db.eps}, ν = {db.min_samples}: {et.max() + 1} clusters, {np.sum(et == -1)} ruido"

def ajustar_gmm(nombre, X):
    # En moons, algunas inicializaciones necesitan más de 500 iteraciones (la mejor no cambia)
    gmm = myGMM(K=hiper[nombre]["K"], n_init=10, max_iter=5000, random_state=0).fit(X)
    return gmm.etiquetas, gmm.mu, f"K = {gmm.K}", gmm.Sigma

metodos = {
    "K-medias": ajustar_kmeans,
    "DBSCAN": ajustar_dbscan,
    "GMM": ajustar_gmm,
}

def grilla_clustering(datasets, metodos, tam=3.6):
    """
    Grilla de gráficos: un conjunto de datos por columna y un método por fila.

    Parámetros:
    - datasets: dict nombre -> X (n, 2)
    - metodos: dict nombre_metodo -> función (nombre_dataset, X) -> (etiquetas, centroides o None, texto[, covarianzas])

    Retorna:
    - fig
    - etiquetas: dict (metodo, nombre_dataset) -> etiquetas
    """
    fig, axes = plt.subplots(len(metodos), len(datasets),
                             figsize=(tam * len(datasets), tam * len(metodos)), squeeze=False)
    etiquetas_todas = {}
    for f, (metodo, ajustar) in enumerate(metodos.items()):
        for col, (nombre, X) in enumerate(datasets.items()):
            etiquetas, centroides, texto, *extra = ajustar(nombre, X)
            etiquetas_todas[(metodo, nombre)] = etiquetas
            ax = axes[f, col]
            graficar_clusters(X, etiquetas, ax, centroides, titulo=texto)
            if extra:
                graficar_elipses(ax, centroides, extra[0])
                ax.set_xlim(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5)
                ax.set_ylim(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5)
            ax.set_xlabel("")
            ax.set_ylabel(metodo if col == 0 else "", fontsize=13)
            if f == 0:
                ax.set_title(f"{nombre}\n{texto}", fontsize=10)
    plt.tight_layout()
    return fig, etiquetas_todas

fig, etiquetas_grilla = grilla_clustering(datasets, metodos)
plt.show()

In [ ]:
ari = {
    nombre: {m: adjusted_rand_score(y_grilla[nombre], etiquetas_grilla[(m, nombre)]) for m in metodos}
    for nombre in datasets
    if y_grilla[nombre] is not None
}
pd.DataFrame(ari).T.round(3)